In [2]:
import re
import pandas as pd
import spacy
from docx import Document

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC


nlp = spacy.load("ru_core_news_lg", disable=["parser", "ner"])

CHAPTER_MAP = {
    "2.2.1": "Общие сведения",
    "2.2.2": "Назначение и цели",
    "2.2.3.1": "Требования к системе",
    "2.2.3.2": "Требования к функциям",
    "2.2.3.3": "Требования к обеспечению",
    "2.2.4": "Этапы работ",
    "2.2.5": "Исполнители работ",
    "2.2.6": "Подготовка к вводу",
    "2.2.7": "Документирование",
}
ALLOWED = set(CHAPTER_MAP.keys())
SECTION_RE = re.compile(r"^\s*(\d+(?:\.\d+)*)(?:\.)?\s+.+$")

def load_dataset(docx_path: str) -> pd.DataFrame:
    doc = Document(docx_path)
    rows, current = [], None
    for p in doc.paragraphs:
        text = (p.text or "").strip()
        if not text:
            continue
        if SECTION_RE.match(text):
            sec = text.split()[0].rstrip(".")
            current = sec if sec in ALLOWED else None
            continue
        if current:
            rows.append({"text": text, "label": current})
    return pd.DataFrame(rows)

df = load_dataset("tz.docx")

# --------- ВАЖНАЯ ПРАВКА №1: убрать редкие классы ---------
MIN_SAMPLES_PER_CLASS = 3  # чтобы можно было делать хотя бы 3-fold CV без предупреждений
counts = df["label"].value_counts()
keep_labels = counts[counts >= MIN_SAMPLES_PER_CLASS].index
df = df[df["label"].isin(keep_labels)].reset_index(drop=True)

print("После фильтра классов (>= 3 примеров):", df.shape)
print(df["label"].value_counts())

# 2) Трансформация (очень простая)
NUM_RE = re.compile(r"\d+")
PUNCT_RE = re.compile(r"[^0-9A-Za-zА-Яа-яЁё\s]+")
WS_RE = re.compile(r"\s+")
LISTNUM_RE = re.compile(r"^\s*\(?\d+(?:\.\d+)*\)?[).]?\s*")

def transform(text, lower=True, mask_numbers=False, remove_punct=False, remove_list_numbers=False):
    t = text
    if lower:
        t = t.lower()
    if remove_list_numbers:
        t = LISTNUM_RE.sub("", t)
    if mask_numbers:
        t = NUM_RE.sub(" NUM ", t)
    if remove_punct:
        t = PUNCT_RE.sub(" ", t)
    return WS_RE.sub(" ", t).strip()

# 3) spaCy: токены -> одна строка
CONTENT_POS = {"NOUN", "ADJ", "VERB", "PROPN"}

def make_clean_text(text, mode):
    if mode == "all":
        text = transform(text, lower=True, remove_list_numbers=True, mask_numbers=True, remove_punct=True)
    else:
        text = transform(text, lower=True, remove_list_numbers=True)

    doc = nlp(text)
    tokens = []
    for tok in doc:
        if tok.is_space or tok.is_punct or tok.is_digit:
            continue
        if mode in ("nostop", "all") and tok.is_stop:
            continue
        if mode in ("pos", "all") and tok.pos_ not in CONTENT_POS:
            continue

        w = tok.lemma_ if mode in ("lemma", "all") else tok.text
        if len(w) >= 2:
            tokens.append(w)

    return " ".join(tokens)

# 4) Конфигурации
CONFIGS = [
    ("base",   dict(min_df=1, max_df=1.0, ngram_range=(1, 1))),
    ("nostop", dict(min_df=1, max_df=1.0, ngram_range=(1, 1))),
    ("lemma",  dict(min_df=1, max_df=1.0, ngram_range=(1, 1))),
    ("pos",    dict(min_df=1, max_df=1.0, ngram_range=(1, 1))),
    ("all",    dict(min_df=2, max_df=0.9, ngram_range=(1, 2))),
]

X = df["text"].astype(str).tolist()
y = df["label"].astype(str).values

# --------- ВАЖНАЯ ПРАВКА №2: подобрать n_splits без предупреждений ---------
min_class = pd.Series(y).value_counts().min()
n_splits = 3 if min_class >= 3 else 2
if n_splits < 2:
    raise ValueError("Слишком мало данных: в каком-то классе меньше 2 примеров.")

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
print("CV folds:", n_splits, "| min_class:", min_class)

results = []
for mode, vec_params in CONFIGS:
    X_clean = [make_clean_text(t, mode) for t in X]

    pipe = make_pipeline(
        TfidfVectorizer(sublinear_tf=True, lowercase=False, **vec_params),
        LinearSVC(class_weight="balanced")
    )

    scores = cross_validate(
        pipe, X_clean, y, cv=cv,
        scoring={"acc": "accuracy", "f1m": "f1_macro"},
        return_train_score=False
    )

    results.append({
        "mode": mode,
        "accuracy": scores["test_acc"].mean(),
        "macro_f1": scores["test_f1m"].mean(),
        "vec": vec_params
    })

res = pd.DataFrame(results).sort_values(["macro_f1", "accuracy"], ascending=False)
display(res)

После фильтра классов (>= 3 примеров): (63, 2)
label
2.2.3.2    15
2.2.3.3    15
2.2.3.1    11
2.2.6      10
2.2.1       6
2.2.4       3
2.2.7       3
Name: count, dtype: int64
CV folds: 3 | min_class: 3


,mode,accuracy,macro_f1,vec
2,lemma,0.619048,0.564793,"{'min_df': 1, 'max_df': 1.0, 'ngram_range': (1..."
4,all,0.650794,0.502220,"{'min_df': 2, 'max_df': 0.9, 'ngram_range': (1..."
1,nostop,0.603175,0.497023,"{'min_df': 1, 'max_df': 1.0, 'ngram_range': (1..."
0,base,0.603175,0.491760,"{'min_df': 1, 'max_df': 1.0, 'ngram_range': (1..."
3,pos,0.587302,0.465701,"{'min_df': 1, 'max_df': 1.0, 'ngram_range': (1..."


1) Таблица с первыми строками датасета (head)

Фрагмент вида:

0, tz.docx, 2.2.1, Общие сведения, Полное наименование системы: …
1, tz.docx, 2.2.1, Общие сведения, Шифр темы: …
…

Это демонстрация того, что мы правильно сделали обучающую выборку:

doc — из какого файла взят текст (у тебя один: tz.docx);

label — класс (глава/раздел: 2.2.1, 2.2.2, 2.2.3.1 …);

label_title — человекочитаемое имя класса (например, “Общие сведения”);

text — конкретный абзац, который является отдельным объектом классификации.

Зачем это в отчёте: показать, что задача “классификация абзацев по разделам ТЗ” формализована корректно, и что text/label получены из файла.

2) Сообщение: “После фильтра классов (>= 3 примеров): (63, 4)”

Это не таблица, а контрольный итог предобработки датасета.

(63, 4) означает: 63 строки (абзаца) и 4 столбца (doc/label/label_title/text).

Фильтр “>=3 примеров на класс” нужен, чтобы:

стратифицированное разбиение/кросс-валидация вообще работали,

модель видела каждый класс хотя бы несколько раз.

Вывод: после фильтра у нас осталось достаточно данных для первичных экспериментов, но 63 — это всё ещё маленький датасет, поэтому мы и перешли на кросс-валидацию.

3) Таблица “config → options” (конфигурации предобработки)

У тебя она выводится строками типа:

B_mask_numbers {'lower': True, 'strip_html': True, 'remove_urls_emails': ...}
C_remove_numbers {...}
A_minimal {...}

Это перечень экспериментальных настроек предобработки — то есть какие именно преобразования текста мы применяли.

Смысл этой таблицы:

чтобы в отчёте было прозрачно: что именно сравнивали (какие методы);

чтобы потом можно было воспроизвести лучший результат.

4) Главная таблица результатов кросс-валидации

Она у тебя выглядит так (по сути):

config	cv_folds	accuracy_mean	accuracy_std	macro_f1_mean	macro_f1_std
B_mask_numbers	3	0.698	0.045	0.555	0.027
C_remove_numbers	3	0.619	0.039	0.518	0.046
A_minimal	3	0.603	0.045	0.492	0.025
D_remove_codes_and_nums	3	0.413	0.192	0.335	0.177
E_remove_section_nums_and_punct	3	0.381	0.169	0.287	0.128
Что означают столбцы

config — имя конфигурации предобработки.

cv_folds = 3 — 3-фолдовая стратифицированная кросс-валидация (мы выбрали 3, потому что в некоторых классах мало примеров; 5 фолдов было бы рискованно).

accuracy_mean — средняя доля правильных ответов на валидации.

accuracy_std — насколько эта точность “плавает” между фолдами.

macro_f1_mean — средний macro-F1 (самая важная метрика при дисбалансе: все классы считаются равноважными).

macro_f1_std — устойчивость macro-F1 между фолдами.

Главные выводы по этой таблице

Лучшая конфигурация — B_mask_numbers.
Она лидирует и по accuracy (~0.70), и по macro-F1 (~0.55). При этом std небольшой → результат стабилен.

Почему “mask_numbers” лучше, чем “remove_numbers”:
В ТЗ числа — это часто информативные признаки (сроки, параметры, номера пунктов, проценты, требования).

Когда мы удаляем числа, мы теряем полезные сигналы.

Когда мы маскируем числа в единый токен NUM, мы сохраняем информацию “здесь было число”, не раздувая словарь кучей конкретных значений.

Почему D и E ухудшили качество:

D_remove_codes_and_nums слишком “агрессивно” режет то, что для ТЗ может быть смысловым (коды, аббревиатуры, числовые требования). Отсюда низкая средняя метрика и огромная нестабильность (std).

E_remove_section_nums_and_punct — ещё более агрессивная чистка: выкидываем и нумерацию, и пунктуацию, и коды. На маленьком датасете это приводит к потере отличительных признаков → качество падает.

В целом качество стало заметно лучше, чем раньше (0.43 macro-F1 → ~0.55 macro-F1).
И это именно потому, что:

предобработка стала “по делу” для жанра ТЗ,

оценка стала устойчивее (CV вместо одного маленького теста).

Итоговые выводы (как в отчёт)

Для данной задачи и данного корпуса наилучшей оказалась предобработка, где числа не удаляются, а маскируются токеном NUM (конфигурация B_mask_numbers).

Слишком агрессивная очистка (удаление кодов/номеров разделов/пунктуации) ухудшает качество: модель теряет характерные маркеры требований, сроков, структурных формулировок.

Macro-F1 предпочтительнее accuracy, потому что классы представлены неравномерно; при этом у лучшей конфигурации macro-F1 устойчив по фолдам (малый std).

KERNEL PYTHON = C:\Users\Mi\AppData\Local\Programs\Python\Python313\python.exe


KERNEL PYTHON = C:\Users\Mi\AppData\Local\Programs\Python\Python313\python.exe
spaCy = 3.8.11
ru_core_news_sm spec = ModuleSpec(name='ru_core_news_sm', loader=<_frozen_importlib_external.SourceFileLoader object at 0x0000020E2475C230>, origin='C:\\Users\\Mi\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages\\ru_core_news_sm\\__init__.py', submodule_search_locations=['C:\\Users\\Mi\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages\\ru_core_news_sm'])


In [1]:
# ============================================
# ЛР1 — Пункт 3: влияние POS и морфологических признаков
# Меняем только spaCy-режим (POS/морфология/леммы), остальное фиксировано
# ============================================

import pandas as pd
import spacy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

# --- spaCy модель (НЕ используем имя sp, чтобы не конфликтовать с scipy as sp) ---
nlp = spacy.load("ru_core_news_lg", disable=["parser", "ner"])
print("spaCy pipeline:", nlp.pipe_names)

# --- фиксируем предобработку (лучший конфиг E из пункта 2) ---
BEST_PREPROCESS = dict(
    lower=True,
    strip_html=True,
    remove_urls_emails=True,
    remove_section_numbers=False,
    normalize_dashes_quotes=True,
    handle_numbers="mask",
    remove_codelike=False,
    keep_punct=False,
    normalize_ws=True
)

X = df["text"].astype(str).values
y = df["label"].astype(str).values
Xp = pd.Series(X).apply(lambda s: preprocess(s, **BEST_PREPROCESS)).values

min_class = pd.Series(y).value_counts().min()
cv = StratifiedKFold(n_splits=5 if min_class >= 5 else 3, shuffle=True, random_state=42)

# --- spaCy -> "строка токенов" (как в примере: токены -> join) ---
def to_mode_text(text, allowed_pos=None, morph=None, repr_mode="text"):
    doc = nlp(text)

    tokens = [t for t in doc if not t.is_space and not t.is_punct and not t.is_digit]

    if allowed_pos:
        tokens = [t for t in tokens if t.pos_ in allowed_pos]

    if morph:
        for feat, allowed_vals in morph.items():
            tokens = [t for t in tokens if set(t.morph.get(feat)).intersection(allowed_vals)]

    if repr_mode == "lemma":
        words = [t.lemma_ for t in tokens]
    elif repr_mode == "lemma_pos":
        words = [f"{t.lemma_}_{t.pos_}" for t in tokens]
    else:
        words = [t.text for t in tokens]

    return " ".join([w for w in words if w])

CONTENT_POS = {"NOUN", "PROPN", "ADJ", "VERB", "ADV"}

MODES = [
    ("BASE_text_allPOS",      dict(allowed_pos=None,        morph=None,                repr_mode="text")),
    ("POS_content_text",      dict(allowed_pos=CONTENT_POS, morph=None,                repr_mode="text")),
    ("POS_noun_text",         dict(allowed_pos={"NOUN"},    morph=None,                repr_mode="text")),
    ("LEMMA_allPOS",          dict(allowed_pos=None,        morph=None,                repr_mode="lemma")),
    ("LEMMA_POS_allPOS",      dict(allowed_pos=None,        morph=None,                repr_mode="lemma_pos")),
    ("MORPH_NOUN_Case=Nom",   dict(allowed_pos={"NOUN"},    morph={"Case": {"Nom"}},   repr_mode="lemma")),
    ("MORPH_VERB_Tense=Past", dict(allowed_pos={"VERB"},    morph={"Tense": {"Past"}}, repr_mode="lemma")),
]

pipe = make_pipeline(
    TfidfVectorizer(sublinear_tf=True, lowercase=False),
    LinearSVC(class_weight="balanced")
)

rows = []
for name, p in MODES:
    X_mode = [to_mode_text(txt, **p) for txt in Xp]

    scores = cross_validate(
        pipe, X_mode, y,
        cv=cv,
        scoring={"acc": "accuracy", "f1m": "f1_macro"},
        return_train_score=False
    )

    rows.append({
        "mode": name,
        "accuracy": scores["test_acc"].mean(),
        "macro_f1": scores["test_f1m"].mean(),
        "params": p
    })

res = pd.DataFrame(rows).sort_values(["macro_f1", "accuracy"], ascending=False)
display(res[["mode", "accuracy", "macro_f1"]])
display(res[["mode", "params"]])

spaCy pipeline: ['tok2vec', 'morphologizer', 'attribute_ruler', 'lemmatizer']


NameError: name 'df' is not defined

In [27]:
# ============================================
# ЛР1 — Пункт 4: нормализация (text vs lemma vs stem)
# Меняем только нормализацию токена, остальное фиксируем.
# ============================================

import pandas as pd
from nltk.stem.snowball import SnowballStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC


# 1) фиксируем лучшую предобработку (E) и готовим данные
BEST_PREPROCESS = dict(
    lower=True, strip_html=True, remove_urls_emails=True,
    remove_section_numbers=False, normalize_dashes_quotes=True,
    handle_numbers="mask", remove_codelike=False,
    keep_punct=False, normalize_ws=True
)

X = df["text"].astype(str).values
y = df["label"].astype(str).values
Xp = pd.Series(X).apply(lambda s: preprocess(s, **BEST_PREPROCESS)).values

min_class = pd.Series(y).value_counts().min()
cv = StratifiedKFold(n_splits=5 if min_class >= 5 else 3, shuffle=True, random_state=42)

# 2) стеммер (как у препода в example.ipynb)
snowball = SnowballStemmer(language="russian")

# 3) токенизатор spaCy с переключением нормализации
def make_tokenizer(mode: str):
    def tok(text: str):
        doc = nlp(text)
        tokens = [t for t in doc if not (t.is_space or t.is_punct)]

        if mode == "text":
            return [t.text if t.text != "NUM" else "NUM" for t in tokens]
        if mode == "lemma":
            return [("NUM" if t.text == "NUM" else t.lemma_) for t in tokens]
        # mode == "stem"
        return [("NUM" if t.text == "NUM" else snowball.stem(t.text)) for t in tokens]
    return tok

# (необязательно, но “по-учебному” показать первые токены как в примере)
sample = Xp[0]
print("TEXT :", make_tokenizer("text")(sample)[:15])
print("LEMMA:", make_tokenizer("lemma")(sample)[:15])
print("STEM :", make_tokenizer("stem")(sample)[:15])

# 4) оценка качества (TF-IDF + LinearSVC), меняем только tokenizer
rows = []
for mode in ["text", "lemma", "stem"]:
    pipe = make_pipeline(
        TfidfVectorizer(
            sublinear_tf=True,
            tokenizer=make_tokenizer(mode),
            token_pattern=None,
            lowercase=False,
        ),
        LinearSVC(class_weight="balanced"),
    )

    scores = cross_validate(
        pipe, Xp, y,
        cv=cv,
        scoring={"acc": "accuracy", "f1m": "f1_macro"},
        return_train_score=False
    )

    rows.append({
        "mode": mode,
        "acc_mean": scores["test_acc"].mean(),
        "acc_std":  scores["test_acc"].std(),
        "f1_mean":  scores["test_f1m"].mean(),
        "f1_std":   scores["test_f1m"].std(),
    })

res = pd.DataFrame(rows).sort_values(["f1_mean", "acc_mean"], ascending=False)
display(res)

best = res.iloc[0]
print(f"Лучший режим: {best['mode']} | macro-F1={best['f1_mean']:.4f}±{best['f1_std']:.4f}, acc={best['acc_mean']:.4f}±{best['acc_std']:.4f}")

TEXT : ['полное', 'наименование', 'системы', 'автоматизированное', 'рабочее', 'место', 'продавца', 'консультанта', 'в', 'салоне', 'фотоуслуг', 'в', 'среде', 'NUM', 'с']
LEMMA: ['полный', 'наименование', 'система', 'автоматизированный', 'рабочий', 'место', 'продавец', 'консультант', 'в', 'салон', 'фотоуслуг', 'в', 'среда', 'NUM', 'с']
STEM : ['полн', 'наименован', 'систем', 'автоматизирова', 'рабоч', 'мест', 'продавц', 'консультант', 'в', 'салон', 'фотоуслуг', 'в', 'сред', 'NUM', 'с']


,mode,acc_mean,acc_std,f1_mean,f1_std
1,lemma,0.666667,0.077762,0.537319,0.033562
2,stem,0.666667,0.077762,0.537319,0.033562
0,text,0.682540,0.097848,0.527115,0.077173


Лучший режим: lemma | macro-F1=0.5373±0.0336, acc=0.6667±0.0778


In [28]:
# ============================================
# ЛР1 — Пункт 5: фильтрация текста и проверка вариантов
# (фиксируем preprocess + lemma + модель + CV)
# меняем только фильтры tokenizer и min_df/max_df
# ============================================

import pandas as pd
import spacy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

# --- 0) spaCy (как в example.ipynb) ---
# если sp уже есть выше — можно эту строку не повторять
sp = spacy.load("ru_core_news_lg", disable=["parser", "ner"])
print("OK:", sp.pipe_names)

# --- 1) фиксируем preprocess (E) и делаем Xp один раз ---
BEST_PREPROCESS = dict(
    lower=True,
    strip_html=True,
    remove_urls_emails=True,
    remove_section_numbers=False,
    normalize_dashes_quotes=True,
    handle_numbers="mask",     # числа -> NUM
    remove_codelike=False,
    keep_punct=False,          # пунктуацию фиксируем
    normalize_ws=True
)

X = df["text"].astype(str).values
y = df["label"].astype(str).values

Xp = df["text"].apply(lambda t: preprocess(t, **BEST_PREPROCESS)).values

min_class = pd.Series(y).value_counts().min()
n_splits = 5 if min_class >= 5 else 3
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
print(f"CV: StratifiedKFold(n_splits={n_splits}), min_samples_per_class={min_class}")

# --- 2) простая tokenizer-функция с фильтрами (лемматизация = фиксирована) ---
def tok(text, use_stop=False, min_len=1, drop_NUM=False, drop_digits=False):
    doc = sp(text)
    out = []
    for t in doc:
        if t.is_space or t.is_punct:
            continue

        raw = t.text
        if drop_NUM and raw == "NUM":
            continue

        lemma = (t.lemma_ or raw).strip()
        if not lemma or lemma == "-PRON-":
            continue

        if use_stop and t.is_stop:
            continue

        if drop_digits and (t.is_digit or any(ch.isdigit() for ch in lemma)):
            continue

        if len(lemma) < min_len:
            continue

        out.append(lemma)
    return out

# --- 3) конфигурации фильтрации ---
FILTERS = [
    {"name": "F0_base",        "use_stop": False, "min_len": 1, "drop_NUM": False, "drop_digits": False, "min_df": 1, "max_df": 1.0},
    {"name": "F1_stopwords",   "use_stop": True,  "min_len": 1, "drop_NUM": False, "drop_digits": False, "min_df": 1, "max_df": 1.0},
    {"name": "F2_len>=2",      "use_stop": False, "min_len": 2, "drop_NUM": False, "drop_digits": False, "min_df": 1, "max_df": 1.0},
    {"name": "F3_len>=3",      "use_stop": False, "min_len": 3, "drop_NUM": False, "drop_digits": False, "min_df": 1, "max_df": 1.0},
    {"name": "F4_drop_NUM",    "use_stop": False, "min_len": 1, "drop_NUM": True,  "drop_digits": False, "min_df": 1, "max_df": 1.0},
    {"name": "F5_min_df=2",    "use_stop": False, "min_len": 1, "drop_NUM": False, "drop_digits": False, "min_df": 2, "max_df": 1.0},
    {"name": "F6_max_df=0.9",  "use_stop": False, "min_len": 1, "drop_NUM": False, "drop_digits": False, "min_df": 1, "max_df": 0.9},
]

# --- 4) оценка конфигураций (метрики + размерность словаря) ---
rows = []
for cfg in FILTERS:
    vectorizer = TfidfVectorizer(
        lowercase=False,
        sublinear_tf=True,
        tokenizer=lambda s, c=cfg: tok(  # фиксируем cfg через дефолт-аргумент
            s,
            use_stop=c["use_stop"],
            min_len=c["min_len"],
            drop_NUM=c["drop_NUM"],
            drop_digits=c["drop_digits"],
        ),
        token_pattern=None,
        min_df=cfg["min_df"],
        max_df=cfg["max_df"],
    )

    pipe = make_pipeline(vectorizer, LinearSVC(class_weight="balanced"))

    scores = cross_validate(
        pipe, Xp, y,
        cv=cv,
        scoring={"acc": "accuracy", "f1m": "f1_macro"},
        return_train_score=False
    )

    # n_features (обучаем один раз на всём корпусе)
    pipe.fit(Xp, y)
    n_features = len(pipe.named_steps["tfidfvectorizer"].get_feature_names_out())

    rows.append({
        "config": cfg["name"],
        "accuracy": scores["test_acc"].mean(),
        "macro_f1": scores["test_f1m"].mean(),
        "n_features": n_features,
        "min_df": cfg["min_df"],
        "max_df": cfg["max_df"],
        "use_stop": cfg["use_stop"],
        "min_len": cfg["min_len"],
        "drop_NUM": cfg["drop_NUM"],
        "drop_digits": cfg["drop_digits"],
    })

res = pd.DataFrame(rows).sort_values(["macro_f1", "accuracy"], ascending=False)
display(res[["config", "accuracy", "macro_f1", "n_features"]])
display(res[["config", "use_stop", "min_len", "drop_NUM", "drop_digits", "min_df", "max_df"]])

best = res.iloc[0]
print(f"BEST: {best['config']} | macro-F1={best['macro_f1']:.4f}, acc={best['accuracy']:.4f}, n_features={int(best['n_features'])}")

OK: ['tok2vec', 'morphologizer', 'attribute_ruler', 'lemmatizer']
CV: StratifiedKFold(n_splits=3), min_samples_per_class=3


,config,accuracy,macro_f1,n_features
1,F1_stopwords,0.730159,0.625259,209
3,F3_len>=3,0.698413,0.607480,230
4,F4_drop_NUM,0.619048,0.550923,253
2,F2_len>=2,0.666667,0.545224,246
0,F0_base,0.666667,0.537319,254
6,F6_max_df=0.9,0.666667,0.537319,254
5,F5_min_df=2,0.555556,0.403666,78


,config,use_stop,min_len,drop_NUM,drop_digits,min_df,max_df
1,F1_stopwords,True,1,False,False,1,1.0
3,F3_len>=3,False,3,False,False,1,1.0
4,F4_drop_NUM,False,1,True,False,1,1.0
2,F2_len>=2,False,2,False,False,1,1.0
0,F0_base,False,1,False,False,1,1.0
6,F6_max_df=0.9,False,1,False,False,1,0.9
5,F5_min_df=2,False,1,False,False,2,1.0


BEST: F1_stopwords | macro-F1=0.6253, acc=0.7302, n_features=209


In [29]:
# ============================================
# ЛР1 — Пункт 6: влияние N-грамм (меняем ТОЛЬКО ngram_range)
# фикс: preprocess=BEST_PREPROCESS, tokenizer=lemma+stopwords, TF-IDF+LinearSVC, StratifiedKFold
# считаем: accuracy, macro-F1, n_features, cv_time, fit_time
# ============================================

import time
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC


# 1) фиксируем данные + разбиение
Xp = df["text"].astype(str).apply(lambda s: preprocess(s, **BEST_PREPROCESS)).values
y  = df["label"].astype(str).values

min_class = pd.Series(y).value_counts().min()
n_splits = 5 if min_class >= 5 else 3
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
print(f"CV: StratifiedKFold, n_splits={n_splits} (min_samples_per_class={min_class})")


# 2) фиксированный токенизатор: lemma + stopwords (как в духе example.ipynb — просто и прямо)
STOP_SET = getattr(nlp.Defaults, "stop_words", set())

def tokenizer_lemma(text: str):
    doc = nlp(text)
    tokens = []
    for t in doc:
        if t.is_space or t.is_punct:
            continue

        if t.text == "NUM":          # NUM оставляем (как у тебя: DROP_NUM=False)
            tokens.append("NUM")
            continue

        lemma = (t.lemma_ or t.text).strip()
        if not lemma or lemma == "-PRON-":
            continue

        if t.is_stop or lemma in STOP_SET:
            continue

        tokens.append(lemma)
    return tokens


# 3) эксперименты только по ngram_range
NGRAMS = [
    ("(1,1)", (1, 1)),
    ("(1,2)", (1, 2)),
    ("(1,3)", (1, 3)),
    ("(2,2)", (2, 2)),  # можно убрать, если не нужно
]

rows = []
for name, rng in NGRAMS:
    tfidf_vectorizer = TfidfVectorizer(
        tokenizer=tokenizer_lemma,
        token_pattern=None,
        lowercase=False,
        sublinear_tf=True,
        ngram_range=rng,
        min_df=1,
        max_df=1.0,
    )
    svc = LinearSVC(class_weight="balanced")
    pipeline = make_pipeline(tfidf_vectorizer, svc)

    t0 = time.perf_counter()
    scores = cross_validate(
        pipeline,
        Xp, y,
        cv=cv,
        scoring={"acc": "accuracy", "f1m": "f1_macro"},
        return_train_score=False
    )
    cv_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    pipeline.fit(Xp, y)
    fit_time = time.perf_counter() - t1

    n_features = len(tfidf_vectorizer.get_feature_names_out())

    rows.append({
        "ngram_range": name,
        "cv_folds": n_splits,
        "accuracy_mean": scores["test_acc"].mean(),
        "accuracy_std":  scores["test_acc"].std(),
        "macro_f1_mean": scores["test_f1m"].mean(),
        "macro_f1_std":  scores["test_f1m"].std(),
        "n_features": n_features,
        "cv_time_sec": cv_time,
        "fit_time_sec": fit_time,
    })

ng_res = pd.DataFrame(rows).sort_values(["macro_f1_mean", "accuracy_mean"], ascending=False)
display(ng_res)

best = ng_res.iloc[0]
print(
    f"ЛУЧШИЙ ngram_range: {best['ngram_range']} | "
    f"macro-F1={best['macro_f1_mean']:.4f}±{best['macro_f1_std']:.4f}, "
    f"acc={best['accuracy_mean']:.4f}±{best['accuracy_std']:.4f}, "
    f"n_features={int(best['n_features'])}, "
    f"cv_time={best['cv_time_sec']:.2f}s"
)

CV: StratifiedKFold, n_splits=3 (min_samples_per_class=3)


,ngram_range,cv_folds,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,n_features,cv_time_sec,fit_time_sec
1,"(1,2)",3,0.730159,0.080937,0.651460,0.085457,494,1.195235,0.422548
0,"(1,1)",3,0.730159,0.059391,0.625259,0.056033,209,1.509480,0.460773
2,"(1,3)",3,0.714286,0.077762,0.601677,0.073197,743,1.207777,0.368566
3,"(2,2)",3,0.380952,0.077762,0.292864,0.060594,285,1.131538,0.372208


ЛУЧШИЙ ngram_range: (1,2) | macro-F1=0.6515±0.0855, acc=0.7302±0.0809, n_features=494, cv_time=1.20s


In [32]:
# ============================================
# ЛР1 — Пункт 7: индексирование (BoW / TF-IDF) + портрет терминов
# (упрощено под стиль example.ipynb) — ИСПРАВЛЕНО
# ============================================

import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_validate
from sklearn.svm import LinearSVC


# --- данные
if df is None or len(df) == 0:
    raise ValueError("df пустой: сначала сформируй датасет (text/label)")

X = df["text"].astype(str).values
y = df["label"].astype(str).values
Xp = df["text"].apply(lambda s: preprocess(s, **BEST_PREPROCESS)).values

TOP_N = 20  # для "портрета"


# --- tokenizer: lemma + stopwords (простая версия)
def tok(text: str):
    doc = nlp(text)
    tokens = []
    for t in doc:
        if t.is_space or t.is_punct:
            continue

        # оставляем NUM как отдельный токен
        if t.text == "NUM":
            tokens.append("NUM")
            continue

        # фильтрация стоп-слов
        if t.is_stop:
            continue

        lemma = (t.lemma_ or "").strip()
        if lemma and lemma != "-PRON-":
            tokens.append(lemma)
    return tokens


# ============================================
# 1) Базовое сравнение BoW vs TF-IDF
# ============================================

bow_vect = CountVectorizer(
    tokenizer=tok,
    token_pattern=None,
    lowercase=False,
    ngram_range=NGRAM_RANGE
)

tfidf_vect = TfidfVectorizer(
    tokenizer=tok,
    token_pattern=None,
    lowercase=False,
    ngram_range=NGRAM_RANGE,
    sublinear_tf=True,
    norm="l2"
)

pipe_bow = make_pipeline(bow_vect, LinearSVC(class_weight="balanced"))
pipe_tfidf = make_pipeline(tfidf_vect, LinearSVC(class_weight="balanced"))

scores_bow = cross_validate(pipe_bow, Xp, y, cv=cv, scoring={"acc": "accuracy", "f1m": "f1_macro"})
scores_tfidf = cross_validate(pipe_tfidf, Xp, y, cv=cv, scoring={"acc": "accuracy", "f1m": "f1_macro"})

# n_features (как у препода: через fit_transform + shape)
bow_mat = sparse.csr_matrix(bow_vect.fit_transform(Xp))
tfidf_mat = sparse.csr_matrix(tfidf_vect.fit_transform(Xp))

base_res = pd.DataFrame([
    {
        "vectorizer": "BoW (CountVectorizer)",
        "accuracy": scores_bow["test_acc"].mean(),
        "macro_f1": scores_bow["test_f1m"].mean(),
        "n_features": bow_mat.shape[1]
    },
    {
        "vectorizer": "TF-IDF",
        "accuracy": scores_tfidf["test_acc"].mean(),
        "macro_f1": scores_tfidf["test_f1m"].mean(),
        "n_features": tfidf_mat.shape[1]
    }
]).sort_values(["macro_f1", "accuracy"], ascending=False)

display(base_res)

# (как в example.ipynb) покажем маленькую матрицу признаков для 2 документов
bow_df = pd.DataFrame(
    bow_mat[:2].toarray(),
    index=["doc1", "doc2"],
    columns=bow_vect.get_feature_names_out()
)
display(bow_df.iloc[:, :20])


# ============================================
# 2) Мини-сетка гиперпараметров (min_df/max_df/max_features)
# ============================================

GRID = [
    (1, 1.0, None),
    (2, 1.0, None),
    (1, 0.9, None),
    (2, 0.9, None),
    (1, 1.0, 300),
    (1, 1.0, 500),
]

rows = []

for kind in ["BoW", "TF-IDF"]:
    for min_df, max_df, max_features in GRID:

        if kind == "BoW":
            vect = CountVectorizer(
                tokenizer=tok, token_pattern=None, lowercase=False,
                ngram_range=NGRAM_RANGE,
                min_df=min_df, max_df=max_df, max_features=max_features
            )
        else:
            vect = TfidfVectorizer(
                tokenizer=tok, token_pattern=None, lowercase=False,
                ngram_range=NGRAM_RANGE,
                min_df=min_df, max_df=max_df, max_features=max_features,
                sublinear_tf=True, norm="l2"
            )

        pipe = make_pipeline(vect, LinearSVC(class_weight="balanced"))
        scores = cross_validate(pipe, Xp, y, cv=cv, scoring={"acc": "accuracy", "f1m": "f1_macro"})

        mat = sparse.csr_matrix(vect.fit_transform(Xp))
        rows.append({
            "vectorizer": kind,
            "min_df": min_df,
            "max_df": max_df,
            "max_features": max_features,           # тут может быть None
            "accuracy": scores["test_acc"].mean(),
            "macro_f1": scores["test_f1m"].mean(),
            "n_features": mat.shape[1]
        })

grid_res = (pd.DataFrame(rows)
            .sort_values(["vectorizer", "macro_f1", "accuracy"], ascending=[True, False, False]))
display(grid_res)

best_tfidf = grid_res[grid_res["vectorizer"] == "TF-IDF"].iloc[0]
print("Лучший TF-IDF по macro-F1:")
display(pd.DataFrame([best_tfidf]))


# ============================================
# 3) TF-IDF "портрет": топ терминов (глобально + по классам)
# ============================================

# --- ВАЖНО: max_features из DataFrame может стать NaN -> приводим к None или int
mf = best_tfidf["max_features"]
if mf is None or pd.isna(mf):
    mf = None
else:
    mf = int(mf)

best_vect = TfidfVectorizer(
    tokenizer=tok, token_pattern=None, lowercase=False,
    ngram_range=NGRAM_RANGE,
    min_df=int(best_tfidf["min_df"]),
    max_df=float(best_tfidf["max_df"]),
    max_features=mf,                 # <-- исправлено
    sublinear_tf=True, norm="l2"
)

X_tfidf = sparse.csr_matrix(best_vect.fit_transform(Xp))
terms = best_vect.get_feature_names_out()

# 3.1 глобально
global_scores = np.asarray(X_tfidf.mean(axis=0)).ravel()
top_idx = global_scores.argsort()[::-1][:TOP_N]
global_portrait = pd.DataFrame({
    "term": terms[top_idx],
    "mean_tfidf": global_scores[top_idx]
})
print("TF-IDF портрет (глобально):")
display(global_portrait)

# 3.2 по классам
class_rows = []
for cls in sorted(pd.unique(y)):
    cls_scores = np.asarray(X_tfidf[y == cls].mean(axis=0)).ravel()
    top_idx = cls_scores.argsort()[::-1][:TOP_N]

    for rank, i in enumerate(top_idx, start=1):
        class_rows.append({
            "label": cls,
            "label_title": CHAPTER_MAP.get(cls, ""),
            "rank": rank,
            "term": terms[i],
            "mean_tfidf_in_class": cls_scores[i]
        })

class_portrait = pd.DataFrame(class_rows)
print("TF-IDF портрет (по классам):")
display(class_portrait.head(60))

,vectorizer,accuracy,macro_f1,n_features
1,TF-IDF,0.730159,0.65146,494
0,BoW (CountVectorizer),0.634921,0.48788,494


,NUM,NUM NUM,NUM hz,NUM me,NUM v,NUM xp,NUM апрель,NUM год,NUM июнь,NUM мб,NUM предприятие,hz,ibm,ibm совместимый,me,me NUM,ms,ms windows,pentium,pentium NUM
doc1,3,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
doc2,4,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


,vectorizer,min_df,max_df,max_features,accuracy,macro_f1,n_features
0,BoW,1,1.0,NaN,0.634921,0.487880,494
2,BoW,1,0.9,NaN,0.634921,0.487880,494
5,BoW,1,1.0,500.0,0.634921,0.487880,494
1,BoW,2,1.0,NaN,0.650794,0.470022,71
3,BoW,2,0.9,NaN,0.650794,0.470022,71
4,BoW,1,1.0,300.0,0.619048,0.438940,300
6,TF-IDF,1,1.0,NaN,0.730159,0.651460,494
8,TF-IDF,1,0.9,NaN,0.730159,0.651460,494
11,TF-IDF,1,1.0,500.0,0.730159,0.651460,494
10,TF-IDF,1,1.0,300.0,0.698413,0.560469,300


Лучший TF-IDF по macro-F1:


,vectorizer,min_df,max_df,max_features,accuracy,macro_f1,n_features
6,TF-IDF,1,1.0,NaN,0.730159,0.65146,494


TF-IDF портрет (глобально):


,term,mean_tfidf
0,NUM,0.059348
1,требование,0.045779
2,справочник,0.043175
3,заказ,0.038066
4,инструкция,0.035352
5,пользователь,0.034244
6,система,0.034205
7,обеспечение,0.025719
8,работа,0.024224
9,NUM NUM,0.022722


TF-IDF портрет (по классам):


,label,label_title,rank,term,mean_tfidf_in_class
0,2.2.1,Общие сведения,1,NUM,0.156445
1,2.2.1,Общие сведения,2,работа,0.101579
2,2.2.1,Общие сведения,3,NUM NUM,0.100202
3,2.2.1,Общие сведения,4,заказчик,0.083667
4,2.2.1,Общие сведения,5,наименование,0.071697
5,2.2.1,Общие сведения,6,порядок,0.062930
6,2.2.1,Общие сведения,7,предприятие,0.062677
7,2.2.1,Общие сведения,8,NUM год,0.060913
8,2.2.1,Общие сведения,9,год,0.060913
9,2.2.1,Общие сведения,10,система,0.060103


In [20]:

import warnings
import numpy as np
import pandas as pd

from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix


warnings.filterwarnings("ignore", category=ConvergenceWarning)

BEST_PREPROCESS = dict(
    lower=True,
    strip_html=True,
    remove_urls_emails=True,
    remove_section_numbers=False,
    normalize_dashes_quotes=True,
    handle_numbers="mask",
    remove_codelike=False,
    keep_punct=False,
    normalize_ws=True
)

X_raw = df["text"].astype(str).values
y = df["label"].astype(str).values
Xp = pd.Series(X_raw).apply(lambda s: preprocess(s, **BEST_PREPROCESS)).values

STOP_SET = set(getattr(nlp.Defaults, "stop_words", set()))

def _tokenize(text: str, repr_mode="lemma", use_stopwords=True, drop_NUM=False, min_len=1):
    doc = nlp(text)
    out = []
    for t in doc:
        if t.is_space or t.is_punct:
            continue

        raw = (t.text or "").strip()
        if not raw:
            continue

        if raw == "NUM":
            if drop_NUM:
                continue
            out.append("NUM")
            continue

        lemma = (t.lemma_ or raw).strip()
        if not lemma or lemma == "-PRON-":
            continue

        if use_stopwords and (t.is_stop or lemma in STOP_SET):
            continue

        if len(lemma) < min_len:
            continue

        if repr_mode == "lemma":
            out.append(lemma)
        elif repr_mode == "lemma_pos":
            out.append(f"{lemma}_{t.pos_}")
        else:
            raise ValueError("repr_mode must be 'lemma' or 'lemma_pos'")
    return out

def tok_lemma_stop(text):     return _tokenize(text, repr_mode="lemma", use_stopwords=True,  drop_NUM=False, min_len=1)
def tok_lemmaPOS_stop(text):  return _tokenize(text, repr_mode="lemma_pos", use_stopwords=True, drop_NUM=False, min_len=1)


min_class = pd.Series(y).value_counts().min()
n_splits = min(5, int(min_class))
if n_splits < 2:
    raise RuntimeError(f"Нельзя делать CV: min_samples_per_class={min_class} < 2")

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
print(f"CV: StratifiedKFold, n_splits={n_splits} (min_samples_per_class={min_class})")

pipe = Pipeline([
    ("vect", TfidfVectorizer(
        lowercase=False,
        tokenizer=tok_lemma_stop,
        token_pattern=None,
        preprocessor=None,
        sublinear_tf=True,
        norm="l2"
    )),
    ("clf", LinearSVC(class_weight="balanced"))
])


def make_logreg():
    return LogisticRegression(
        class_weight="balanced",
        solver="saga",
        max_iter=20000,
    )



param_grid = [
    {
        "vect__tokenizer": [tok_lemma_stop, tok_lemmaPOS_stop],
        "vect__ngram_range": [(1,1), (1,2), (1,3)],
        "vect__min_df": [1, 2],
        "vect__max_df": [1.0, 0.9],
        "vect__max_features": [None, 500, 300],
        "clf": [LinearSVC(class_weight="balanced")],
        "clf__C": [0.5, 1.0, 2.0, 5.0],
    },

    {
        "vect__tokenizer": [tok_lemma_stop],
        "vect__ngram_range": [(1,1), (1,2)],
        "vect__min_df": [1, 2],
        "vect__max_df": [1.0, 0.9],
        "vect__max_features": [None, 500],
        "clf": [make_logreg()],
        "clf__C": [0.5, 1.0, 2.0, 5.0],
    },

    {
        "vect__tokenizer": [tok_lemma_stop],
        "vect__ngram_range": [(1,1), (1,2)],
        "vect__min_df": [1, 2],
        "vect__max_df": [1.0, 0.9],
        "vect__max_features": [None, 500],
        "clf": [MultinomialNB()],
        "clf__alpha": [0.1, 0.5, 1.0],
    },

    {
        "vect": [CountVectorizer(
            lowercase=False,
            tokenizer=tok_lemma_stop,
            token_pattern=None,
            preprocessor=None
        )],
        "vect__ngram_range": [(1,1), (1,2)],
        "vect__min_df": [1, 2],
        "vect__max_df": [1.0, 0.9],
        "vect__max_features": [None, 500],
        "clf": [MultinomialNB()],
        "clf__alpha": [0.1, 0.5, 1.0],
    },
]

search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    refit=True,
    n_jobs=None,
    verbose=0,
    error_score=np.nan,
    return_train_score=False
)

search.fit(Xp, y)

print("Best CV macro-F1:", search.best_score_)
print("Best params (short):")
for k, v in search.best_params_.items():
    if k in ("vect", "clf"):
        print(f"  {k}: {type(v).__name__}")
    elif k == "vect__tokenizer":
        print(f"  {k}: {'tok_lemmaPOS_stop' if v==tok_lemmaPOS_stop else 'tok_lemma_stop'}")
    else:
        print(f"  {k}: {v}")

best_model = search.best_estimator_

vect = best_model.named_steps["vect"]
try:
    n_features = len(vect.get_feature_names_out())
except Exception:
    n_features = len(getattr(vect, "vocabulary_", {}))
print("Best model n_features:", n_features)

y_pred_oof = cross_val_predict(best_model, Xp, y, cv=cv)

print("\n=== OOF (CV) classification_report ===")
print(classification_report(y, y_pred_oof, digits=4, zero_division=0))

labels_sorted = sorted(pd.unique(y))
cm = confusion_matrix(y, y_pred_oof, labels=labels_sorted)
cm_df = pd.DataFrame(cm, index=[f"true:{l}" for l in labels_sorted], columns=[f"pred:{l}" for l in labels_sorted])

print("\n=== OOF (CV) confusion matrix ===")
display(cm_df)

print("\nLegend (label -> title):")
for l in labels_sorted:
    print(f"{l} -> {CHAPTER_MAP.get(l, '')}")

cv_res = pd.DataFrame(search.cv_results_)
top = cv_res.sort_values("mean_test_score", ascending=False).head(10)[
    ["mean_test_score", "std_test_score", "rank_test_score", "params"]
].reset_index(drop=True)

print("\n=== TOP-10 configs by CV macro-F1 ===")
display(top)

CV: StratifiedKFold, n_splits=3 (min_samples_per_class=3)
Best CV macro-F1: 0.677764034906892
Best params (short):
  clf: LogisticRegression
  clf__C: 5.0
  vect__max_df: 0.9
  vect__max_features: None
  vect__min_df: 1
  vect__ngram_range: (1, 2)
  vect__tokenizer: tok_lemma_stop
Best model n_features: 494

=== OOF (CV) classification_report ===
              precision    recall  f1-score   support

       2.2.1     0.7500    0.5000    0.6000         6
     2.2.3.1     1.0000    0.6364    0.7778        11
     2.2.3.2     1.0000    0.9333    0.9655        15
     2.2.3.3     0.6000    0.8000    0.6857        15
       2.2.4     0.7500    1.0000    0.8571         3
       2.2.6     0.9091    1.0000    0.9524        10
       2.2.7     0.0000    0.0000    0.0000         3

    accuracy                         0.7778        63
   macro avg     0.7156    0.6957    0.6912        63
weighted avg     0.8070    0.7778    0.7781        63


=== OOF (CV) confusion matrix ===


,pred:2.2.1,pred:2.2.3.1,pred:2.2.3.2,pred:2.2.3.3,pred:2.2.4,pred:2.2.6,pred:2.2.7
true:2.2.1,3,0,0,2,1,0,0
true:2.2.3.1,1,7,0,3,0,0,0
true:2.2.3.2,0,0,14,0,0,1,0
true:2.2.3.3,0,0,0,12,0,0,3
true:2.2.4,0,0,0,0,3,0,0
true:2.2.6,0,0,0,0,0,10,0
true:2.2.7,0,0,0,3,0,0,0



Legend (label -> title):
2.2.1 -> Общие сведения
2.2.3.1 -> Требования к системе
2.2.3.2 -> Требования к функциям
2.2.3.3 -> Требования к обеспечению
2.2.4 -> Этапы работ
2.2.6 -> Подготовка к вводу
2.2.7 -> Документирование

=== TOP-10 configs by CV macro-F1 ===


,mean_test_score,std_test_score,rank_test_score,params
0,0.677764,0.048508,1,{'clf': LogisticRegression(class_weight='balan...
1,0.671786,0.065264,2,"{'clf': MultinomialNB(), 'clf__alpha': 0.1, 'v..."
2,0.671786,0.065264,2,"{'clf': MultinomialNB(), 'clf__alpha': 0.1, 'v..."
3,0.671786,0.065264,2,"{'clf': MultinomialNB(), 'clf__alpha': 0.1, 'v..."
4,0.671786,0.065264,2,"{'clf': MultinomialNB(), 'clf__alpha': 0.1, 'v..."
5,0.666252,0.064644,6,"{'clf': LinearSVC(class_weight='balanced'), 'c..."
6,0.666252,0.064644,6,"{'clf': LinearSVC(class_weight='balanced'), 'c..."
7,0.666252,0.064644,6,"{'clf': LinearSVC(class_weight='balanced'), 'c..."
8,0.666252,0.064644,6,"{'clf': LinearSVC(class_weight='balanced'), 'c..."
9,0.663469,0.069926,10,"{'clf': MultinomialNB(), 'clf__alpha': 0.1, 'v..."
